## import necessary libraries

In [6]:
!pip install lightgbm


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix

In [8]:
credit_card_data = pd.read_csv("credit_card_clean.csv")
credit_card_data

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,1,20000.0,female,university,married,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,female,university,single,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,female,university,single,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,female,university,married,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,male,university,married,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000.0,male,highschool,married,39,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,29997,150000.0,male,highschool,single,43,-1,-1,-1,-1,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,29998,30000.0,male,university,single,37,4,3,2,-1,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,29999,80000.0,male,highschool,married,41,1,-1,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


## Data Understanding

In [9]:
credit_card_data.isna().sum()

ID           0
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
DEFAULT      0
dtype: int64

In [11]:
credit_card_data.shape

(30000, 25)

In [12]:
credit_card_data.dtypes

ID             int64
LIMIT_BAL    float64
SEX           object
EDUCATION     object
MARRIAGE      object
AGE            int64
PAY_1          int64
PAY_2          int64
PAY_3          int64
PAY_4          int64
PAY_5          int64
PAY_6          int64
BILL_AMT1    float64
BILL_AMT2    float64
BILL_AMT3    float64
BILL_AMT4    float64
BILL_AMT5    float64
BILL_AMT6    float64
PAY_AMT1     float64
PAY_AMT2     float64
PAY_AMT3     float64
PAY_AMT4     float64
PAY_AMT5     float64
PAY_AMT6     float64
DEFAULT        int64
dtype: object

## Data preparation

In [22]:
del credit_card_data["ID"]

In [23]:
le = LabelEncoder()
credit_card_data["SEX"] = le.fit_transform(credit_card_data["SEX"])
credit_card_data["EDUCATION"] = le.fit_transform(credit_card_data["EDUCATION"])
credit_card_data["MARRIAGE"] = le.fit_transform(credit_card_data["MARRIAGE"])

## model building

In [50]:
# 1. AdaBoost: Needs shallow base trees (depth 1 to 3) and moderate estimators
abc = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)

# 2. Gradient Boosting: Shallow trees prevent overfitting; slower learning rate
gbc = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# 3. XGBoost: Can handle slightly deeper trees due to strong internal regularization
xgbc = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    random_state=42
)

# 4. LightGBM: Uses num_leaves for tree complexity; max_depth prevents extreme growth
lgbc = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=31,
    random_state=42
)

In [51]:
X = credit_card_data.drop("DEFAULT",axis = 1)
y = credit_card_data["DEFAULT"]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.20,random_state = 123,shuffle = True,stratify = y)

## model training

In [52]:
%%time
abc.fit(X_train,y_train)

CPU times: total: 7.81 s
Wall time: 7.8 s


,estimator,None
,n_estimators,200
,learning_rate,0.1
,algorithm,'deprecated'
,random_state,42


In [53]:
%%time
gbc.fit(X_train,y_train)

CPU times: total: 38.6 s
Wall time: 38.7 s


,loss,'log_loss'
,learning_rate,0.05
,n_estimators,300
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,4
,min_impurity_decrease,0.0
,init,None


In [54]:
%%time
xgbc.fit(X_train,y_train)

CPU times: total: 14.9 s
Wall time: 2.05 s


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [55]:
%%time
lgbc.fit(X_train,y_train)

[LightGBM] [Info] Number of positive: 5309, number of negative: 18691
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002939 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3259
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.221208 -> initscore=-1.258639
[LightGBM] [Info] Start training from score -1.258639
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,8
,learning_rate,0.03
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## model testing

In [57]:
y_pred_abc = abc.predict(X_test)
y_pred_gbc = gbc.predict(X_test)
y_pred_xgbc = xgbc.predict(X_test)
y_pred_lgbc =  lgbc.predict(X_test)

## model evaluation

## adaboost

In [60]:
print("Accuracy Score:",round(accuracy_score(y_test,y_pred_abc),4))
print("Precision Score:",round(precision_score(y_test,y_pred_abc),4))
print("Recall Score",round(recall_score(y_test,y_pred_abc)),4)
print("Confusion Matrix\n",confusion_matrix(y_test,y_pred_abc))

Accuracy Score: 0.8177
Precision Score: 0.6886
Recall Score 0 4
Confusion Matrix
 [[4436  237]
 [ 848  479]]


## gradient boosting

In [61]:
print("Accuracy Score:",round(accuracy_score(y_test,y_pred_gbc),4))
print("Precision Score:",round(precision_score(y_test,y_pred_gbc),4))
print("Recall Score",round(recall_score(y_test,y_pred_gbc)),4)
print("Confusion Matrix\n",confusion_matrix(y_test,y_pred_gbc))

Accuracy Score: 0.8228
Precision Score: 0.6886
Recall Score 0 4
Confusion Matrix
 [[4455  218]
 [ 845  482]]


## extreme gradient boosting

In [62]:
print("Accuracy Score:",round(accuracy_score(y_test,y_pred_xgbc),4))
print("Precision Score:",round(precision_score(y_test,y_pred_xgbc),4))
print("Recall Score",round(recall_score(y_test,y_pred_xgbc)),4)
print("Confusion Matrix\n",confusion_matrix(y_test,y_pred_xgbc))

Accuracy Score: 0.82
Precision Score: 0.6722
Recall Score 0 4
Confusion Matrix
 [[4438  235]
 [ 845  482]]


## light gradient boosting

In [63]:
print("Accuracy Score:",round(accuracy_score(y_test,y_pred_lgbc),4))
print("Precision Score:",round(precision_score(y_test,y_pred_lgbc),4))
print("Recall Score",round(recall_score(y_test,y_pred_lgbc)),4)
print("Confusion Matrix\n",confusion_matrix(y_test,y_pred_lgbc))

Accuracy Score: 0.8192
Precision Score: 0.669
Recall Score 0 4
Confusion Matrix
 [[4436  237]
 [ 848  479]]
